# S01 · From Molecules to Labeled Graphs: Fundamentals

## Aim of this talktorial

Reaction modeling via **graph transformation** (e.g. double-pushout rules) rests on two fundamental pillars:

1. **Representation** — encoding molecules as graphs with chemically meaningful node and edge labels.
2. **Matching** — determining when two graphs (or parts of graphs) correspond via **labeled graph morphisms**, including:
   - **isomorphisms** (exact equivalence),
   - **automorphisms** (internal symmetry).

This talktorial introduces both pillars through **minimal, explicit implementations** that make the underlying assumptions transparent. We combine:

- **RDKit** as the chemical reference layer (sanitization, aromaticity, valence semantics),
- **NetworkX** as a general-purpose graph engine for matching, symmetry analysis, and later graph rewriting.

The goal is not performance, but **conceptual clarity**: to expose how chemical choices in labeling and symmetry handling directly affect downstream tasks such as rule extraction and application in later SynEdu notebooks.

---

## Learning outcomes

After completing this talktorial, you will be able to:

- Convert SMILES strings into **labeled molecular graphs** (atoms as nodes, bonds as edges).
- Perform a **round-trip conversion** between RDKit and NetworkX representations and identify what chemical information is preserved or lost.
- Formally define a **labeled graph morphism** and recognize its key special cases:
  - homomorphism,
  - monomorphism,
  - isomorphism,
  - automorphism.
- Distinguish between:
  - **graph isomorphism** (structural equivalence under relabeling),
  - **graph automorphisms** (symmetries of a single molecule).
- Contrast **chemistry-aware matching** in RDKit (SMARTS-based) with **structure-and-attribute matching** in NetworkX.
- Explain why **symmetry** and **label design** play a central role in reaction rule discovery and application later in SynEdu.

---

## Outline

- [0. Setup & data](#0-setup--data)
- [1. Labeled molecular graphs](#1-labeled-molecular-graphs)
- [2. Graph isomorphism](#2-graph-isomorphism)
- [3. Graph automorphisms](#3-graph-automorphisms)
- [4. Discussion](#4-discussion)
- [5. Quiz](#5-quiz)
- [6. References](#6-references)


## 0. Setup & Data

In [2]:
import rdkit
import networkx as nx
import pandas as pd
from pathlib import Path

print("RDKit version:", rdkit.__version__)
print("NetworkX version:", nx.__version__)

DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "molecules.csv"
df = pd.read_csv(CSV_PATH)
display(df)

RDKit version: 2025.09.3
NetworkX version: 3.6.1


,smiles,name
0,CCO,ethanol
1,CCCl,chloroethane
2,CCOCC,diethyl ether
3,O,water
4,NCC(=O)O,glycine
...,...,...
995,CNCC(O)c1ccc(O)cc1,oxedrine
996,Cc1nc(Nc2ccc(F)cc2)nc(N2CCc3ccccc3C2C)c1C,revaprazan
997,O=C(O)CCC(=O)c1ccc2c(c1)-c1cccc3cccc-2c13,florantyrone
998,COC(=O)CCC[N+](C)(C)C,carpronium


## 1. Labeled Molecular Graphs

In computational reaction modeling, we represent molecules as **labeled graphs** so that any notion of
“matching” respects **chemical identity**—such as element type, charge, and bond order—rather than bare
connectivity alone.

A **labeled molecular graph** is a quadruple

$$
G = (V, E, \tau_V, \tau_E),
$$

where:

- **Vertices** $V$ represent **atoms**.
- **Edges** $E \subseteq \{\{u,v\}\mid u,v\in V,\ u\neq v\}$ represent **bonds**
  (finite, undirected, simple: no loops, no parallel edges).
- $\tau_V: V \to \mathcal{A}_V$ assigns **atom attributes** (chemical labels).
- $\tau_E: E \to \mathcal{A}_E$ assigns **bond attributes** (chemical labels).

We often write $V(G)$ and $E(G)$ for the vertex and edge sets of $G$. For a vertex $v\in V(G)$:

- neighbourhood:
  $$
  N_G(v)=\{w\in V(G)\mid vw\in E(G)\},
  $$
- degree:
  $$
  \deg_G(v)=|N_G(v)|.
  $$

These graph-theoretic notions correspond chemically to an atom’s bonded neighbours and coordination number,
abstracting away geometry while retaining connectivity.

---

### Labels and chemical types

“Types” are encoded via explicit labelling maps

$$
\ell_V: V(G)\to L_V,\qquad \ell_E: E(G)\to L_E,
$$

where $L_V$ and $L_E$ are finite, non-empty label sets.
For molecular graphs, we use the chemistry-specific notation:

$$
a_G: V(G)\to L_V \quad\text{(atom labels)},\qquad
b_G: E(G)\to L_E \quad\text{(bond labels)}.
$$

Let $\mathcal{G}$ denote the class of all labelled molecular graphs equipped with $(a_G,b_G)$.
In chemistry, $a_G(v)$ encodes *what atom this is* (element, charge, aromaticity, hydrogen count, …), 
while $b_G(uv)$ encodes *what bond this is* (order, aromaticity, ring status, …).

All subsequent notions of equivalence, symmetry, and matching in this talktorial are defined **relative to these labels**.

---

### Graph representations in practice

In SynEdu, **RDKit** and **NetworkX** play complementary roles:

- **RDKit** is the chemical authority: sanitization, valence rules, aromaticity perception, and canonicalization.
- **NetworkX** provides an explicit, inspectable graph representation used for matching, symmetry analysis,
  and later graph rewriting.

To ensure that graph-based operations remain chemically meaningful, we require a **reversible interface**
between the two representations:
a molecule converted from RDKit to a labeled NetworkX graph must be convertible back without loss of
the chemical information encoded in $(a_G,b_G)$.

This reversible interface forms the foundation for all later notions—graph isomorphism, automorphisms,
and eventually reaction rules—introduced in subsequent SynEdu notebooks.


In [3]:
from typing import Dict
import networkx as nx
from rdkit import Chem
import rdkit

In [4]:
# RDKit -> NetworkX
def mol_to_graph(mol: Chem.Mol, include_implicit_h: bool = True) -> nx.Graph:
    """
    Convert RDKit Mol -> labeled NetworkX graph.

    :param mol: RDKit Mol (assumed sanitized).
    :param include_implicit_h: If True, store total H count per atom as ``total_h``.
    :returns: networkx.Graph with atom/bond labels as node/edge attributes.
    """
    G = nx.Graph()

    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        attrs: Dict[str, object] = {
            "symbol": atom.GetSymbol(),
            "formal_charge": int(atom.GetFormalCharge()),
            "aromatic": bool(atom.GetIsAromatic()),
            "chiral_tag": str(atom.GetChiralTag()),
        }
        if include_implicit_h:
            attrs["total_h"] = int(atom.GetTotalNumHs())
        G.add_node(i, **attrs)

    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        order = int(round(bond.GetBondTypeAsDouble()))
        G.add_edge(
            u,
            v,
            order=order,
            aromatic=bool(bond.GetIsAromatic()),
            in_ring=bool(bond.IsInRing()),
        )

    G.graph["source"] = "rdkit"
    G.graph["rdkit_version"] = rdkit.__version__
    return G

In [5]:
# NetworkX to rdkit
def graph_to_mol(G: nx.Graph, make_explicit_h: bool = False) -> Chem.Mol:
    """
    Reconstruct RDKit Mol from labeled NetworkX graph (inverse of ``mol_to_graph`` up to sanitization).

    :param G: labeled molecular graph produced by ``mol_to_graph``.
    :param make_explicit_h: If True and ``total_h`` exists, add explicit H atoms (best-effort).
    :returns: Sanitized RDKit Mol.
    """
    rw = Chem.RWMol()
    nx_to_rdk: Dict[int, int] = {}

    # atoms
    for node in sorted(G.nodes()):
        n = G.nodes[node]
        atom = Chem.Atom(n.get("symbol", "C"))
        atom.SetFormalCharge(int(n.get("formal_charge", 0)))
        if n.get("aromatic", False):
            atom.SetIsAromatic(True)

        ch_tag = n.get("chiral_tag")
        if ch_tag and ch_tag != "CHI_UNSPECIFIED":
            try:
                atom.SetChiralTag(getattr(Chem.rdchem.ChiralType, ch_tag))
            except Exception:
                pass  # best-effort only

        nx_to_rdk[node] = rw.AddAtom(atom)

    # bonds
    for u, v, e in G.edges(data=True):
        order = int(e.get("order", 1))
        btype = {
            1: Chem.rdchem.BondType.SINGLE,
            2: Chem.rdchem.BondType.DOUBLE,
            3: Chem.rdchem.BondType.TRIPLE,
        }.get(order, Chem.rdchem.BondType.SINGLE)
        rw.AddBond(nx_to_rdk[u], nx_to_rdk[v], btype)

    mol = rw.GetMol()

    # optional explicit H
    if make_explicit_h:
        for node, rdk_idx in nx_to_rdk.items():
            total_h = G.nodes[node].get("total_h")
            if total_h is None:
                continue
            atom = mol.GetAtomWithIdx(rdk_idx)
            current_h = sum(1 for n in atom.GetNeighbors() if n.GetSymbol() == "H")
            for _ in range(max(int(total_h) - current_h, 0)):
                h_idx = mol.AddAtom(Chem.Atom("H"))
                mol.AddBond(rdk_idx, h_idx, Chem.rdchem.BondType.SINGLE)

    Chem.SanitizeMol(mol)
    return mol

### Exercise: Round-trip accuracy (RDKit ⇄ NetworkX)

The goal of this exercise is to verify that converting

RDKit → NetworkX → RDKit

preserves the **chemical information we care about**.

You should treat the two functions provided above as a black box.

---

#### Q1 — Heavy-atom SMILES invariance

Write a function `roundtrip_smiles_equal(smiles)` that:

1. parses a SMILES string into an RDKit molecule,
2. converts it to a labeled graph using `mol_to_graph`,
3. reconstructs a molecule using `graph_to_mol`,
4. compares the **canonical heavy-atom SMILES** of the original and reconstructed molecules.

The function should return `True` if the two SMILES are identical, and `False` otherwise.

---

#### Q2 — Count invariants

Extend your check in **Q1** to also verify that:

- the number of **heavy atoms** is preserved,
- the number of **heavy-atom bonds** is preserved.

Return `True` only if *all* invariants are satisfied.

> Hint: use `Chem.RemoveHs(mol)` before counting atoms or bonds.

---


<details>
<summary><b>Solution:</b></summary>

### Q1–Q2: Round-trip checker (heavy SMILES + count invariants)

```python
from rdkit import Chem
from rdkit.Chem import rdmolops  # optional: useful for extra invariants

def canonical_heavy_smiles(m: Chem.Mol) -> str:
    """Return canonical SMILES after removing H (heavy-atom skeleton)."""
    return Chem.MolToSmiles(Chem.RemoveHs(m), canonical=True)

def heavy_counts(m: Chem.Mol) -> tuple[int, int]:
    """Return (n_heavy_atoms, n_heavy_bonds) after removing H."""
    mh = Chem.RemoveHs(m)
    return mh.GetNumAtoms(), mh.GetNumBonds()

def roundtrip_ok(smiles: str, verbose: bool = True) -> bool:
    """
    RDKit → NetworkX → RDKit round-trip check.

    Criteria:
    1) canonical heavy-atom SMILES preserved
    2) heavy atom count preserved
    3) heavy bond count preserved
    """
    m1 = Chem.MolFromSmiles(smiles)
    if m1 is None:
        if verbose:
            print("Parse failed:", smiles)
        return False

    G = mol_to_graph(m1, include_implicit_h=True)
    m2 = graph_to_mol(G, make_explicit_h=False)

    s1, s2 = canonical_heavy_smiles(m1), canonical_heavy_smiles(m2)
    c1, c2 = heavy_counts(m1), heavy_counts(m2)

    ok = (s1 == s2) and (c1 == c2)

    if verbose and not ok:
        print("FAIL:", smiles)
        print(" heavy SMILES:", s1, "vs", s2)
        print(" counts:", c1, "vs", c2)

    return ok
```

### Quick test (run on a small subset)

```python
n_test = min(50, len(df))
fails = []

for s in df["smiles"].head(n_test):
    if not roundtrip_ok(s, verbose=False):
        fails.append(s)

print("Checked:", n_test)
print("Failures:", len(fails))
if fails:
    print("Example failures:", fails[:5])

# Optional: inspect one failure in detail
if fails:
    _ = roundtrip_ok(fails[0], verbose=True)
```


## 2. Graph isomorphism

To make “same molecule” precise, we model molecules as labeled graphs and compare them via
**label-preserving maps**. This section introduces **labeled graph morphisms** and the induced notion of
**graph isomorphism**.

---

### 2.1 Graph morphisms

Let $G,H \in \mathcal{G}$ be labeled molecular graphs with atom/bond labeling functions
$(a_G,b_G)$ and $(a_H,b_H)$.

A **(labeled) graph morphism** from $G$ to $H$ is a map

$$
\varphi : V(G) \to V(H)
$$

that preserves atom types and bond structure. Concretely, for all $v \in V(G)$ and $uv \in E(G)$:

**(M1) Atom-label preservation**
$$
a_H(\varphi(v)) = a_G(v).
$$

**(M2) Adjacency preservation**
$$
\varphi(u)\varphi(v) \in E(H).
$$

**(M3) Bond-label preservation**
$$
b_H\!\big(\varphi(u)\varphi(v)\big) = b_G(uv).
$$

> **Chemist’s view**  
> $\varphi$ can only rename atom indices: it cannot change element/charge (M1), remove bonds (M2),
> or change bond types (M3), insofar as these are encoded in $(a_\cdot,b_\cdot)$.

---

### 2.2 Graph isomorphism

Two labeled molecular graphs $G,H\in\mathcal{G}$ are **isomorphic**, written

$$
G \cong H,
$$

if there exists a **bijective** morphism

$$
\varphi : V(G) \to V(H)
$$

satisfying (M1)–(M3), whose inverse $\varphi^{-1}$ also satisfies (M1)–(M3).

> **Chemist’s view**  
> $G \cong H$ means the two graphs encode the *same chemical structure* up to a renumbering of atoms.

### 2.3. Practice

In practice, `networkx` tests $G \cong H$ by searching for such a bijection under the chosen
`node_match` / `edge_match`. The result therefore depends on the label scheme (and any compatibility rules).

$$
\Phi_V : V(G)\times V(H)\to\{\text{true},\text{false}\},
\qquad
\Phi_E : E(G)\times E(H)\to\{\text{true},\text{false}\},
$$

Throughout this talktorials, we use a *strict-but-minimal* label model:

- atom: `symbol`, `formal_charge`, `aromatic`,
- bond: `order`.

This keeps the equivalence relation explicit and reproducible; later notebooks revisit and relax these choices.


In [7]:
from rdkit import Chem
from networkx.algorithms import isomorphism as iso

pairs = {
    "benzene": ("c1ccccc1", "C1=CC=CC=C1"),
    "aniline": ("c1ccccc1N", "c1ccccc1[NH3+]"),
}

graphs = {}
for name, (sa, sb) in pairs.items():
    graphs[f"{name}_a"] = mol_to_graph(Chem.MolFromSmiles(sa))
    graphs[f"{name}_b"] = mol_to_graph(Chem.MolFromSmiles(sb))


def node_match(n1, n2):
    return n1.get("symbol") == n2.get("symbol")


def edge_match(e1, e2):
    return int(e1.get("order", 1)) == int(e2.get("order", 1))


def iso_and_count(G1, G2, nm, em):
    gm = iso.GraphMatcher(G1, G2, node_match=nm, edge_match=em)
    return gm.is_isomorphic(), sum(1 for _ in gm.isomorphisms_iter())


print("=== simple matcher (symbol + order) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]
    G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")

=== simple matcher (symbol + order) ===
benzene  | isomorphic: 1 | mappings: 12
aniline  | isomorphic: 1 | mappings: 2


### Q3 — Isomorphism

Implement `node_match` that requires matching `symbol` **and** either `total_h` or `formal_charge` (or both). Replace the existing `node_match` with your function and re-run the demo so that:

- `benzene` still matches, and  
- `aniline` (`c1ccccc1N`) **does not** match `anilinium` (`c1ccccc1[NH3+]`).

> Hint: `mol_to_graph(..., include_implicit_h=True)` stores H as `total_h`. Use `n.get("total_h",0)` or `n.get("formal_charge",0)`.



<details> <summary><b>Solution:</b></summary>

```python
# Solution: enhanced matcher that checks symbol + (total_h OR formal_charge)
def enhanced_node_match(n1, n2):
    return (
        n1.get("symbol") == n2.get("symbol")
        and (
            int(n1.get("total_h", 0)) == int(n2.get("total_h", 0))
            or int(n1.get("formal_charge", 0)) == int(n2.get("formal_charge", 0))
        )
    )

# run the demo with the enhanced matcher (uses existing `pairs`, `graphs`, `edge_match`, `iso_and_count`)
print("=== enhanced matcher (symbol + total_h/charge) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]; G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, enhanced_node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")

# quick instructor checks
assert iso_and_count(graphs["benzene_a"], graphs["benzene_b"], enhanced_node_match, edge_match)[0]
assert not iso_and_count(graphs["aniline_a"], graphs["aniline_b"], enhanced_node_match, edge_match)[0]
```
<details>

## 3. Graph automorphisms

**Observation.** In the benzene example you enumerated **12 mappings** — these are the automorphisms of the benzene heavy-atom graph (the dihedral group \(D_6\), where \(|D_6| = 12\)).

**Definition.** An automorphism is a graph isomorphism from the graph to itself:

$$
f : G \longrightarrow G.
$$

The automorphism group is

$$
\mathrm{Aut}(G).
$$

The **orbit** of a vertex \(v\) is the set of images of \(v\) under all automorphisms:

$$
\mathrm{Orbit}(v)=\{\psi(v)\;|\;\psi\in\mathrm{Aut}(G)\}.
$$

**Facts.** For benzene:

$$
|\mathrm{Aut}(G)| = |D_6| = 12,
$$

and all six carbon atoms lie in a single orbit.

**Why it matters.** Symmetric hosts produce many equivalent embeddings → duplicate matches and wasted work.

**Simple remedies.**
- Deduplicate by host-atom set: use `frozenset(mapping.values())`.  
- Use orbit representatives (e.g. choose the $\min$ index per orbit).  
- Accept only a canonical mapping (WL/lexicographic tie-break).

**Practical tips.**
- Include chemical attributes (`total_h`, `formal_charge`, stereochemistry) in matchers to reduce false symmetry.  
- Pre-filter with cheap signatures (degree, label counts, WL hashes) before enumerating automorphisms.



### Q4: Automorphisms of a Molecular Graph

Task: Develop a function `enumerate_automorphisms` to enumerate all automorphisms of a graph

**Hint:** An automorphism of a graph \(G\) is an isomorphism from \(G\) to itself.

Equivalently, the automorphism group satisfies

$$
\mathrm{Aut}(G) \subseteq \mathrm{Iso}(G, G).
$$



<details> <summary><b>Solution:</b></summary>

```python
def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())
```

In [7]:
# Instructor utility (used later): enumerate automorphisms via NetworkX GraphMatcher
# You can treat this as the computational counterpart of "Aut(G)" in the theory section.

from networkx.algorithms import isomorphism as iso


def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())

We can now analyse the symmetry of a molecular graph by computing the
**orbits induced by its automorphism group**.

Under the natural action of the automorphism group on the vertex set,
two vertices belong to the same orbit if there exists an automorphism
mapping one to the other.

$$
\text{For } u, v \in V(G), \quad
u \sim v
\;\Longleftrightarrow\;
\exists\, \varphi \in \mathrm{Aut}(G)
\text{ such that }
\varphi(u) = v .
$$

Each orbit therefore represents a set of **symmetry-equivalent atoms**.


In [8]:
import networkx as nx
from typing import Dict, Iterable, List, Set


def compute_orbits_from_automorphisms(
    G: nx.Graph,
    automorphisms: Iterable[Dict] | None = None,
) -> List[Set]:
    """
    Compute vertex orbits induced by the automorphism group of a graph.

    Given the automorphism group Aut(G) acting on V(G), two vertices
    u, v ∈ V(G) belong to the same orbit if there exists an automorphism
    φ ∈ Aut(G) such that φ(u) = v.

    This function computes the orbits by collapsing vertices connected
    by automorphism mappings using a union–find (disjoint-set) structure.

    Parameters
    ----------
    G : nx.Graph
        Input graph.
    automorphisms : iterable of dict, optional
        Precomputed automorphisms φ : V(G) → V(G).
        If None, they are computed internally.

    Returns
    -------
    List[Set]
        List of vertex orbits. Each orbit is a set of nodes.
        Ordering is deterministic (sorted by smallest element).
    """
    if automorphisms is None:
        automorphisms = enumerate_automorphisms(G)

    # --- Disjoint-set (union–find) structure ---
    parent: Dict = {v: v for v in G.nodes()}

    def find(x):
        """Find representative with path compression."""
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a, b):
        """Union sets containing a and b."""
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # --- Apply group action ---
    for auto in automorphisms:
        for v, fv in auto.items():
            union(v, fv)

    # --- Collect orbits ---
    orbits: Dict = {}
    for v in G.nodes():
        r = find(v)
        orbits.setdefault(r, set()).add(v)

    # deterministic ordering (useful for teaching & testing)
    return sorted(orbits.values(), key=lambda s: min(s))


from rdkit import Chem

benzene = Chem.MolFromSmiles("c1ccccc1")
G_bz = mol_to_graph(benzene)

autos = enumerate_automorphisms(G_bz)
orbits = compute_orbits_from_automorphisms(G_bz, autos)

print("Number of automorphisms (benzene):", len(autos))
print("Orbits:", orbits)

Number of automorphisms (benzene): 12
Orbits: [{0, 1, 2, 3, 4, 5}]


## 4. Discussion
- A **labeled graph morphism** formalizes structure- and attribute-preserving maps between graphs.
- Our **labeled molecular graphs** use a minimal attribute schema (`symbol`, `formal_charge`, `aromatic`, `order`) to define what “same” means.
- **Round-trip conversion** (RDKit → NetworkX → RDKit) is valuable for debugging and peer review; we preserve heavy-atom topology, but exact RDKit internal state may differ.
- **Automorphisms** describe symmetries; they inflate match enumeration. Deduplicate (e.g. by host-atom set) to prevent combinatorial explosion.

## 5. Quiz

Answer the following questions using **both chemical intuition and formal graph language**.

---

### 1. labeled graph morphism

In one or two sentences, define a **labeled graph morphism**.

$$
f : G \rightarrow H
$$

- What objects does f map?
- Which **atom** and **bond** properties must be preserved?
- Give one example of a mapping that would be **invalid** in chemistry.


---

### 2. Isomorphism  
What **additional requirement** must a graph morphism satisfy to become an  
**isomorphism**?

- How does this relate to the idea of *two molecules having the same structure*?

---

### 3. Automorphism and symmetry  
What is an **automorphism** of a molecular graph?

- Why do symmetric molecules (e.g. benzene) have **many automorphisms**?
- Why do automorphisms cause **duplicate subgraph matches** during matching?

---

### 4. Deduplicating subgraph matches  
Subgraph matching often returns many equivalent matches.

- Explain how using **sets of host atom indices** can be used to
  **deduplicate** equivalent matches.
- Why does this work even when atom ordering differs?

---

### 5. RDKit vs NetworkX (practice vs theory)  
Give **one practical advantage** of each approach:

- **RDKit** substructure matching  
- **NetworkX** graph matching  

In which situations would you prefer one over the othe


## 6. References

- RDKit documentation: https://www.rdkit.org/docs/  
- RDKit Book: https://www.rdkit.org/docs/Book.html  
- NetworkX documentation: https://networkx.org/documentation/stable/  
- NetworkX isomorphism: https://networkx.org/documentation/stable/reference/algorithms/isomorphism.html  
- RDKit MCS (rdFMCS): https://www.rdkit.org/docs/source/rdkit.Chem.rdFMCS.html  
